# Notebook 5: Combined Results Summary
Generates all final comparison charts from experiment CSVs.

In [ ]:
!pip install matplotlib pandas -q

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
import os

os.makedirs("results/figures", exist_ok=True)

df_q = pd.read_csv("results/tables/quant_results.csv")
df_w = pd.read_csv("results/tables/window_results.csv")
df_p = pd.read_csv("results/tables/perplexity_results.csv")
df_e = pd.read_csv("results/tables/energy_results.csv")

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
df_512 = df_q[df_q["seq_len"] == 512]
baseline = df_512[df_512["quant"] == "FP16"]["peak_mem_mb"].values[0]
reductions = [(baseline - row["peak_mem_mb"]) / baseline * 100 for _, row in df_512.iterrows()]
bars = ax1.bar(df_512["quant"], reductions, color=['#3498db','#e67e22','#e74c3c'])
ax1.set_title("Memory Reduction vs FP16 (seq=512)")
ax1.set_ylabel("Memory Reduction (%)")
for bar, val in zip(bars, reductions):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.1f}%", ha='center')
ax1.grid(True, alpha=0.3, axis='y')

ax2 = fig.add_subplot(gs[0, 1])
for quant, grp in df_q.groupby("quant"):
    ax2.plot(grp["seq_len"], grp["throughput_tok_s"], marker='o', label=quant)
ax2.set_title("Throughput vs Sequence Length")
ax2.set_xlabel("Sequence Length")
ax2.set_ylabel("Tokens / Second")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 0])
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db']
bars = ax3.bar(df_w["window"], df_w["mem_reduction_pct"], color=colors)
ax3.set_title("KV Cache Reduction — Sliding Window")
ax3.set_ylabel("Memory Reduction (%)")
ax3.set_ylim(0, 100)
for bar, val in zip(bars, df_w["mem_reduction_pct"]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{val}%", ha='center')
ax3.grid(True, alpha=0.3, axis='y')

ax4 = fig.add_subplot(gs[1, 1])
df_e512 = df_e[df_e["seq_len"] == 512]
bars = ax4.bar(df_e512["quant"], df_e512["energy_uj"], color=['#3498db','#e67e22','#e74c3c'])
ax4.set_title("Energy per Token at seq=512 (μJ)")
ax4.set_ylabel("Energy (microjoules)")
for bar, val in zip(bars, df_e512["energy_uj"]):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"{val:.1f}", ha='center')
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle("Memory-Centric LLM Optimizations — Combined Results", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("results/figures/combined_results.png", dpi=150)
print("Saved combined_results.png")
plt.show()